In [1]:
import torch

from datasets import load_dataset
from torch.utils.data import Dataset

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.13.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
dataset = load_dataset(
    "imagefolder",
    data_dir=r"C:\Users\praut\project crop doctor\Rice"
)

print(dataset)
print("Columns:", dataset["train"].column_names)

Resolving data files:   0%|          | 0/4078 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 4078
    })
})
Columns: ['image', 'label']


In [3]:
print("Total Rice images:", len(dataset["train"]))

Total Rice images: 4078


In [4]:
dataset = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

print("Training images:", len(dataset["train"]))
print("Validation images:", len(dataset["test"]))

Training images: 3262
Validation images: 816


In [5]:
labels = dataset["train"].features["label"].names

print("Rice classes:")
for i, label in enumerate(labels):
    print(i, ":", label)

Rice classes:
0 : Rice___Brown_Spot
1 : Rice___Healthy
2 : Rice___Leaf_Blast
3 : Rice___Neck_Blast


In [6]:
id2label = {
    i: label for i, label in enumerate(labels)
}

label2id = {
    label: i for i, label in enumerate(labels)
}

print("Number of classes:", len(labels))

Number of classes: 4


In [7]:
model_name = "google/vit-base-patch16-224-in21k"

processor = ViTImageProcessor.from_pretrained(
    model_name
)

print("ViT processor loaded successfully!")

ViT processor loaded successfully!


In [8]:
class RiceDataset(Dataset):

    def __init__(self, hf_dataset, processor):
        self.dataset = hf_dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        item = self.dataset[idx]

        image = item["image"].convert("RGB")

        inputs = self.processor(
            images=image,
            return_tensors="pt"
        )

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "labels": item["label"]
        }

In [9]:
train_dataset = RiceDataset(
    dataset["train"],
    processor
)

eval_dataset = RiceDataset(
    dataset["test"],
    processor
)

print("Rice datasets created successfully!")

Rice datasets created successfully!


In [10]:
sample = train_dataset[0]

print(sample.keys())
print("Image shape:", sample["pixel_values"].shape)
print("Label:", sample["labels"])

dict_keys(['pixel_values', 'labels'])
Image shape: torch.Size([3, 224, 224])
Label: 3


In [11]:
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("Rice ViT model created successfully!")

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.attention.output.dense.bias      | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.bias     | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.weight    | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.bias   | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.weight           | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.bias            | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.bias          | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.weight   | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.weight        | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.bias   

Rice ViT model created successfully!


In [12]:
def collate_fn(examples):

    pixel_values = torch.stack([
        example["pixel_values"]
        for example in examples
    ])

    labels_batch = torch.tensor([
        example["labels"]
        for example in examples
    ])

    return {
        "pixel_values": pixel_values,
        "labels": labels_batch
    }

In [13]:
training_args = TrainingArguments(
    output_dir="./rice-vit-results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,

    logging_steps=50,

    report_to="none"
)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    data_collator=collate_fn
)

print("Trainer created successfully!")

Trainer created successfully!


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.653967,0.654593
2,0.661452,0.602098
3,0.550320,0.603038


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1224, training_loss=0.6508956555447547, metrics={'train_runtime': 960.168, 'train_samples_per_second': 10.192, 'train_steps_per_second': 1.275, 'total_flos': 7.583502038307471e+17, 'train_loss': 0.6508956555447547, 'epoch': 3.0})

In [16]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch
0.550320,0.602098,3


{'eval_loss': 0.6020979881286621}


In [17]:
trainer.save_model("./rice-vit-final")
processor.save_pretrained("./rice-vit-final")

print("🌾 Rice model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🌾 Rice model saved successfully!
